# ContactEase — Rubrica di contatti

**ContactEase Solutions** — applicazione console interattiva per gestire una
rubrica di contatti telefonici.

Il progetto applica i principi della **programmazione orientata agli oggetti**
in Python e permette di:

| # | Funzionalità |
|---|---|
| 1 | Aggiungere un contatto |
| 2 | Visualizzare tutti i contatti |
| 3 | Cercare un contatto per nome o cognome |
| 4 | Modificare un contatto esistente |
| 5 | Eliminare un contatto |
| 6 | Salvare i contatti su file (formato **JSON**) |
| | Caricare automaticamente i contatti all'avvio |

## Come è organizzato il notebook

1. Introduzione (questa cella)
2. Installazione libreria e import
3. – 4. Classe `Contatto`
5. – 8. Classe `Rubrica` (dati, ricerca, persistenza)
9. – 10. Funzioni dell'interfaccia (libreria `rich`)
11. – 12. Funzione `main()` con il menu
13. Avvio dell'applicazione
14. Test automatici

**Esegui le celle in ordine dall'alto verso il basso**, poi usa la cella di
avvio per far partire la rubrica.

## Nota sulla persistenza dei dati su Google Colab

I contatti vengono salvati nel file `contatti.json` nella cartella di lavoro
di Colab. Questa cartella **viene azzerata quando la sessione si chiude**: i
dati restano quindi disponibili finché la sessione è attiva.

Per conservare i contatti in modo permanente puoi montare Google Drive e
spostare il file lì, cambiando `NOME_FILE` più avanti nel notebook:

```python
from google.colab import drive
drive.mount('/content/drive')
NOME_FILE = '/content/drive/MyDrive/contatti.json'
```

## Import e installazione

Installiamo la libreria `rich` (serve per un'interfaccia a riga di comando
più leggibile: pannelli, tabelle, colori) e importiamo tutto ciò che serve.
Su Google Colab `rich` è quasi sempre già installata: il comando non farà
nulla in quel caso.

In [1]:
%pip install rich

import json
import os
import time
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from IPython.display import clear_output


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\carmi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


## La classe `Contatto`

Rappresenta **un singolo contatto** della rubrica. È una classe semplice: tiene
insieme i dati di una persona e sa convertirsi da/verso un dizionario, il
formato usato per salvare su file JSON.

Attributi:

- `id` — numero identificativo, assegnato dalla `Rubrica` e **stabile** nel tempo
- `nome`, `cognome`, `numero` — dati obbligatori
- `email`, `indirizzo` — dati facoltativi (possono essere stringhe vuote)

Metodi:

- `to_dict()` — restituisce un dizionario con i campi del contatto
- `from_dict(d)` — *metodo di classe*: ricostruisce un `Contatto` da un dizionario

In [2]:
class Contatto:
    """Rappresenta un singolo contatto della rubrica."""

    def __init__(self, id, nome, cognome, numero, email, indirizzo):
        self.id = id
        self.nome = nome
        self.cognome = cognome
        self.numero = numero
        self.email = email
        self.indirizzo = indirizzo

    def to_dict(self):
        """Converte il contatto in un dizionario pronto per il JSON."""
        return {
            "id": self.id,
            "nome": self.nome,
            "cognome": self.cognome,
            "numero": self.numero,
            "email": self.email,
            "indirizzo": self.indirizzo,
        }

    @classmethod
    def from_dict(cls, d):
        """Ricostruisce un Contatto da un dizionario letto dal JSON."""
        return cls(d["id"], d["nome"], d["cognome"],
                    d["numero"], d["email"], d["indirizzo"])

## La classe `Rubrica`

È il **cuore dell'applicazione**: conserva i contatti e si occupa di tutte
le operazioni, incluso il salvataggio su file. Non stampa nulla a schermo (a
parte un avviso se il file non c'è o è danneggiato): la parte grafica è
tenuta separata, nelle funzioni dell'interfaccia (più avanti).

Stato interno:

- `percorso_file` — nome del file JSON dei contatti
- `contatti` — **dizionario**: chiave = `id` del contatto, valore = oggetto
  `Contatto`
- `prossimo_id` — prossimo `id` libero da assegnare

Perché un dizionario e non una lista? Perché la ricerca per `id` (usata da
`modifica` ed `elimina`) diventa un accesso diretto — `self.contatti.get(id)`
— invece di dover scorrere tutti i contatti uno per uno. Quando i dati hanno
una chiave naturale e la cerchi spesso per quella chiave, il dizionario è la
struttura giusta.

Per **id** il dizionario ci dà quindi un accesso diretto. Ma per cercare "chi
si chiama Mario" non c'è una chiave pronta: bisogna scorrere tutti i valori
del dizionario (`self.contatti.values()`) e controllare uno per uno. È il
prezzo da pagare quando si cerca per qualcosa di diverso dalla chiave — ma
dato che una rubrica personale ha al massimo qualche centinaio di contatti,
scorrerli tutti resta comunque velocissimo.

Metodi:

| Metodo | Cosa fa |
|---|---|
| `aggiungi(...)` | crea un nuovo contatto con l'`id` corrente, lo inserisce nel dizionario e aggiorna il contatore |
| `trova_per_id(id)` | restituisce il contatto con quell'`id`, oppure `None` |
| `modifica(id, ...)` | aggiorna i campi di un contatto; `True` se esiste |
| `elimina(id)` | rimuove un contatto; `True` se esisteva |
| `cerca(testo)` | contatti il cui **nome o cognome** contiene `testo` (maiuscole/minuscole indifferenti) |
| `elenco()` | tutti i contatti ordinati per cognome, poi nome |
| `is_vuota()` | `True` se non ci sono contatti |
| `salva()` | scrive i contatti nel file JSON |
| `carica()` | legge i contatti dal file JSON, se esiste ed è valido |

### Formato del file `contatti.json`

```json
{
  "prossimo_id": 3,
  "contatti": [
    {"id": 1, "nome": "Mario", "cognome": "Rossi",
     "numero": "+39 333 1234567", "email": "mario@example.com",
     "indirizzo": "Via Roma 1, Milano"}
  ]
}
```

Il contatore `prossimo_id` è salvato nel file: così gli `id` restano stabili
anche dopo eliminazioni e riavvii dell'applicazione. Se il file manca o è
danneggiato, `carica()` non solleva un errore: avvisa e parte con una
rubrica vuota, senza toccare il file esistente.

In [3]:
class Rubrica:
    """Gestisce i contatti (in un dizionario id -> Contatto) e la loro persistenza."""

    def __init__(self, percorso_file):
        self.percorso_file = percorso_file
        self.contatti = {}      # chiave: id (int) -> valore: oggetto Contatto
        self.prossimo_id = 1

    def aggiungi(self, nome, cognome, numero, email, indirizzo):
        """Crea un nuovo contatto, lo inserisce nel dizionario e lo restituisce."""
        contatto = Contatto(self.prossimo_id, nome, cognome, numero, email, indirizzo)
        self.contatti[contatto.id] = contatto
        self.prossimo_id += 1
        return contatto

    def trova_per_id(self, id):
        """Restituisce il contatto con quell'id, oppure None (accesso diretto)."""
        return self.contatti.get(id)

    def modifica(self, id, nome, cognome, numero, email, indirizzo):
        """Aggiorna i campi del contatto con quell'id. True se esiste."""
        contatto = self.trova_per_id(id)
        if contatto is None:
            return False
        contatto.nome = nome
        contatto.cognome = cognome
        contatto.numero = numero
        contatto.email = email
        contatto.indirizzo = indirizzo
        return True

    def elimina(self, id):
        """Rimuove il contatto con quell'id. True se esisteva."""
        if id not in self.contatti:
            return False
        del self.contatti[id]
        return True

    def cerca(self, testo):
        """Contatti il cui nome O cognome contiene 'testo' (case-insensitive)."""
        testo = testo.strip().lower()
        if testo == "":
            return []
        return [c for c in self.contatti.values()
                if testo in c.nome.lower() or testo in c.cognome.lower()]

    def elenco(self):
        """Tutti i contatti ordinati per cognome, poi nome."""
        return sorted(self.contatti.values(),
                      key=lambda c: (c.cognome.lower(), c.nome.lower()))

    def is_vuota(self):
        """True se non ci sono contatti."""
        return len(self.contatti) == 0

    def salva(self):
        """Scrive contatti e contatore su file JSON (UTF-8, leggibile)."""
        dati = {
            "prossimo_id": self.prossimo_id,
            "contatti": [c.to_dict() for c in self.contatti.values()],
        }
        with open(self.percorso_file, "w", encoding="utf-8") as f:
            json.dump(dati, f, indent=2, ensure_ascii=False)

    def carica(self):
        """Carica i contatti dal file JSON, se esiste ed e' valido."""
        if not os.path.exists(self.percorso_file):
            print(f"File '{self.percorso_file}' non trovato: rubrica vuota.")
            return
        try:
            with open(self.percorso_file, encoding="utf-8") as f:
                dati = json.load(f)
        except (json.JSONDecodeError, OSError):
            print(f"File '{self.percorso_file}' illeggibile o corrotto: "
                  f"rubrica vuota (il file non e' stato modificato).")
            return
        contatti_letti = [Contatto.from_dict(d) for d in dati.get("contatti", [])]
        self.contatti = {c.id: c for c in contatti_letti}
        if "prossimo_id" in dati:
            self.prossimo_id = dati["prossimo_id"]
        elif self.contatti:
            self.prossimo_id = max(self.contatti.keys()) + 1
        else:
            self.prossimo_id = 1

## Le funzioni dell'interfaccia

Sono funzioni separate dalle classi: si occupano **solo** di mostrare cose a
schermo e di leggere l'input dell'utente, usando la libreria `rich` per una
resa ordinata (pannelli, tabelle, colori).

### Principio: una schermata per volta

L'applicazione gira dentro l'output di **una sola cella** (quella con
`main()`, più avanti). Ogni volta che l'utente fa una scelta, quell'output
viene **ripulito** prima di mostrare il passo successivo. A schermo restano
solo:

1. l'**intestazione** dell'azione corrente;
2. le **informazioni necessarie in quel momento** — senza residui dei passi
   precedenti.

`intestazione(titolo)` fa proprio questo: pulisce lo schermo e stampa il
pannello del titolo.

### Come vengono lette le risposte

Tutte le domande passano da `chiedi_riga(messaggio)`, che stampa la domanda
nell'**output della cella** (così è visibile, non solo nella casella di
input del frontend) e poi legge una riga da tastiera. Il piccolo `flush` e
la pausa prima di `input()` evitano un difetto noto di Jupyter/Colab per cui
la prima `input()` dopo un `clear_output()` restituirebbe subito una
stringa vuota, senza aspettare che l'utente digiti.

| Funzione | Ruolo |
|---|---|
| `pulisci_schermo()` | svuota l'output della cella |
| `intestazione(titolo)` | pulisce lo schermo e stampa il pannello del titolo |
| `chiedi_riga(messaggio)` | stampa la domanda e legge una riga da tastiera |
| `mostra_menu()` | elenca le voci del menu |
| `mostra_tabella(contatti, titolo)` | mostra i contatti in tabella |
| `chiedi_dati_contatto(correnti=None)` | chiede i 5 campi di un contatto |
| `leggi_intero(messaggio)` | legge un numero, ripetendo se l'input non è valido |
| `messaggio_ok / messaggio_errore / messaggio_info` | righe colorate di esito |
| `pausa()` | attende la pressione di Invio |

In [4]:
# Un unico oggetto Console condiviso da tutte le funzioni di stampa.
console = Console()

# Voci del menu principale (posizione + 1 = numero mostrato a video).
VOCI_MENU = [
    "Aggiungi contatto",
    "Visualizza tutti i contatti",
    "Cerca contatto (per nome o cognome)",
    "Modifica contatto",
    "Elimina contatto",
    "Salva su file",
    "Esci",
]


def pulisci_schermo():
    """Svuota l'output della cella (Colab/Jupyter)."""
    clear_output(wait=True)


def intestazione(titolo):
    """Pulisce lo schermo e stampa il pannello dell'azione corrente."""
    pulisci_schermo()
    console.print(Panel(f"[bold]{titolo}[/bold]",
                        title="[bold cyan]ContactEase[/bold cyan]",
                        border_style="cyan", box=box.DOUBLE))


def chiedi_riga(messaggio=""):
    """Stampa 'messaggio' nell'output della cella e legge una riga da tastiera."""
    print(messaggio, end="", flush=True)
    time.sleep(0.1)
    return input()


def mostra_menu():
    """Elenca le voci numerate del menu principale."""
    for numero, voce in enumerate(VOCI_MENU, start=1):
        console.print(f"  [bold yellow]{numero}[/bold yellow]) {voce}")
    console.print()


def mostra_tabella(contatti, titolo):
    """Mostra i contatti in una tabella; se vuota, un messaggio informativo."""
    if not contatti:
        messaggio_info("Nessun contatto da mostrare.")
        return
    tabella = Table(title=titolo, box=box.SIMPLE_HEAVY, title_style="bold")
    for colonna in ("ID", "Nome", "Cognome", "Numero", "Email", "Indirizzo"):
        tabella.add_column(colonna)
    for c in contatti:
        tabella.add_row(str(c.id), c.nome, c.cognome, c.numero, c.email, c.indirizzo)
    console.print(tabella)


def _chiedi_campo(etichetta, obbligatorio, valore_corrente=None):
    """Chiede un singolo campo.

    - Invio (risposta vuota) mantiene 'valore_corrente' se fornito (modifica).
    - Se il campo e' facoltativo e la risposta e' vuota, restituisce "".
    - Se e' obbligatorio e non c'e' un valore corrente, ripete la domanda.
    """
    suffisso = f" [{valore_corrente}]" if valore_corrente not in (None, "") else ""
    while True:
        risposta = chiedi_riga(f"{etichetta}{suffisso}: ").strip()
        if risposta == "" and valore_corrente is not None:
            return valore_corrente
        if risposta == "" and not obbligatorio:
            return ""
        if risposta != "":
            return risposta
        console.print("  [red]Campo obbligatorio.[/red]")


def chiedi_dati_contatto(correnti=None):
    """Raccoglie i 5 campi di un contatto e li restituisce in un dizionario."""
    c = correnti or {}
    return {
        "nome":      _chiedi_campo("Nome", True, c.get("nome")),
        "cognome":   _chiedi_campo("Cognome", True, c.get("cognome")),
        "numero":    _chiedi_campo("Numero", True, c.get("numero")),
        "email":     _chiedi_campo("Email", False, c.get("email")),
        "indirizzo": _chiedi_campo("Indirizzo", False, c.get("indirizzo")),
    }


def leggi_intero(messaggio):
    """Legge un numero intero, ripetendo la domanda finche' non e' valido."""
    while True:
        risposta = chiedi_riga(f"{messaggio}: ").strip()
        if risposta == "":
            continue
        try:
            return int(risposta)
        except ValueError:
            console.print("  [red]Inserisci un numero intero.[/red]")


def messaggio_ok(testo):
    console.print(f"[bold green]OK[/bold green] {testo}")


def messaggio_errore(testo):
    console.print(f"[bold red]Errore[/bold red] {testo}")


def messaggio_info(testo):
    console.print(f"[cyan]{testo}[/cyan]")


def pausa():
    chiedi_riga("\nPremi Invio per continuare...")